# 03_01 — Specialist Ensemble Training

Train one specialist model per ToxiGen group and configure the ensemble voting weights.

**Design**
- Each specialist is fine-tuned with plain cross-entropy on its group's data only (no density weighting during training).
- Density weights are used exclusively at voting time.
- The K value and embedding space (raw / PCA) for the ensemble weights should be chosen based on the best general model configuration found in **notebook 04, section 3**.

**Sections**
1. Configuration
2. Train all specialists
3. Specialist metrics summary
4. Configure ensemble weights (pick K + space)

In [ ]:
import sys, os
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

import json
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.training import TrainingConfig
from src.training.specialist import train_all_specialists, TOXIGEN_GROUPS
from src.evaluation.ensemble import compute_group_weights, normalise_weights

sns.set_theme(style='whitegrid')

## 1. Configuration

In [ ]:
CONFIG_PATH = 'configs/datasets.yaml'
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

EMBEDDINGS_ROOT   = cfg['embedding']['output_root']
PREPROCESSED_ROOT = cfg['preprocessing']['output_root']
TRAINING_ROOT     = cfg['training']['output_root']
MODEL_NAME        = cfg['embedding']['models'][0]
MODEL_SLUG        = MODEL_NAME.replace('/', '_')
K_VALUES          = cfg['embedding']['k_values']

train_cfg = cfg['training']

base_config = TrainingConfig(
    model_id      = train_cfg['models'][0],
    batch_size    = train_cfg['batch_size'],
    learning_rate = train_cfg['learning_rate'],
    num_epochs    = train_cfg['epochs'],
    max_length    = train_cfg['max_length'],
    random_state  = train_cfg['random_state'],
    output_root   = train_cfg['output_root'],
    density_column= None,   # specialists use plain CE
)

TOXIGEN_DENSITIES_CSV = os.path.join(EMBEDDINGS_ROOT, 'toxigen', MODEL_SLUG, 'densities.csv')

print('Model         :', base_config.model_id)
print('Epochs        :', base_config.num_epochs)
print('Groups        :', TOXIGEN_GROUPS)
print('Densities CSV :', TOXIGEN_DENSITIES_CSV)

## 2. Train All Specialists

One model per ToxiGen group, plain cross-entropy loss.  
Set `RUN_TRAINING = True` the first time. Subsequent runs load cached `metrics.json`.

In [ ]:
RUN_TRAINING = False  # set True to train (slow)

if RUN_TRAINING:
    all_metrics = train_all_specialists(
        density_csv  = TOXIGEN_DENSITIES_CSV,
        config       = base_config,
        groups       = TOXIGEN_GROUPS,
        skip_existing= True,
    )
else:
    # Load cached metrics
    all_metrics = {}
    for group in TOXIGEN_GROUPS:
        path = os.path.join(
            TRAINING_ROOT, base_config.model_slug(),
            f'{group}__specialist', 'metrics.json'
        )
        if os.path.exists(path):
            with open(path) as f:
                all_metrics[group] = json.load(f)
        else:
            print(f'[MISSING] {group} — run with RUN_TRAINING=True')

print(f'Loaded metrics for {len(all_metrics)} specialists')

## 3. Specialist Metrics Summary

In [ ]:
if all_metrics:
    rows = [
        {
            'group'            : g,
            'f1'               : m.get('f1', float('nan')),
            'balanced_accuracy': m.get('balanced_accuracy', float('nan')),
            'accuracy'         : m.get('accuracy', float('nan')),
            'auc_roc'          : m.get('auc_roc', float('nan')),
        }
        for g, m in all_metrics.items()
    ]
    summary = pd.DataFrame(rows).sort_values('f1', ascending=False).set_index('group')
    display(
        summary.style
        .format('{:.4f}')
        .background_gradient(subset=['f1'], cmap='YlGn')
    )
else:
    print('No metrics yet — train first.')

In [ ]:
if all_metrics:
    fig, ax = plt.subplots(figsize=(10, 5))
    groups_sorted = summary.index.tolist()
    ax.bar(groups_sorted, summary.loc[groups_sorted, 'f1'], color='#3498db')
    ax.set_xlabel('Group')
    ax.set_ylabel('F1')
    ax.set_title('Specialist model F1 per group (eval on held-out split)')
    ax.set_xticklabels(groups_sorted, rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 4. Configure Ensemble Weights

Choose `K` and `SPACE` based on the best general model configuration from **notebook 04**.  
The table below shows the pre-computed group weights for your chosen configuration.

In [ ]:
# --- Set these based on the best general model in notebook 04 ---
K     = 100    # best K value from general model evaluation
SPACE = 'raw'  # 'raw' or 'pca'

In [ ]:
raw_weights = compute_group_weights(TOXIGEN_DENSITIES_CSV, k=K, space=SPACE)
norm_weights = normalise_weights(raw_weights)

weights_df = (
    pd.DataFrame([
        {'group': g, 'raw_weight': raw_weights[g], 'normalised_weight': norm_weights[g]}
        for g in sorted(raw_weights)
    ])
    .set_index('group')
    .sort_values('normalised_weight', ascending=False)
)

display(weights_df.style.format('{:.4f}').background_gradient(subset=['normalised_weight'], cmap='YlOrRd'))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
groups_w = weights_df.index.tolist()
ax.bar(groups_w, weights_df['normalised_weight'], color='#e67e22')
ax.set_xlabel('Group')
ax.set_ylabel('Normalised weight')
ax.set_title(f'Ensemble voting weights — K={K}, space={SPACE}')
ax.set_xticklabels(groups_w, rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f'\nEnsemble config to use in notebook 04:')
print(f'  K     = {K}')
print(f'  SPACE = "{SPACE}"')